# CV4IS Übungsprojekt — Part 1  
## CNN from Scratch for Binary Industrial Defect Classification

In this project, you will build a complete image-classification pipeline for a simple industrial visual inspection task.

**Task:** classify hazelnut images as:

```text
0 = good
1 = defective
```

For this first part, you must train a **CNN from scratch**.  
Pretrained models are **not allowed** for Part 1.

This notebook is intentionally only lightly guided. The goal is to combine what you learned in previous exercises into one reproducible industrial computer vision pipeline.

## Dataset download and preparation

This notebook uses a Hazelnut binary classification dataset (based on MvTec AD) for training and evaluation.  
To keep the Git repository small, the dataset is **not stored directly in the repository**. Instead, it is downloaded automatically from a password-protected public sciebo link.

The cell below performs the following steps:

1. Creates the local `data/` directory if it does not already exist.
2. Downloads the dataset zip file from sciebo into `data/`.
3. Shows a progress bar during the download.
4. Extracts the zip file into the `data/` directory.
5. Verifies that the expected dataset folders are available:

   - `data/hazelnut_binary_raw`
   - `data/hazelnut_binary_corruptions`

If the dataset has already been downloaded and extracted, the cell skips the unnecessary steps. This makes the notebook easier to rerun without repeatedly downloading the same file.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import zipfile

import requests
from tqdm.auto import tqdm


SCIEBO_URL = "https://ruhr-uni-bochum.sciebo.de/s/6SnpA9egsirgbgo"
SCIEBO_PASSWORD = "CV4IS_SS2026"

DATA_DIR = Path.cwd() / "data"
ZIP_PATH = DATA_DIR / "hazelnut_dataset.zip"

EXPECTED_DATASET_DIRS = [
    DATA_DIR / "hazelnut_binary_raw",
    DATA_DIR / "hazelnut_binary_corruptions",
]

print(f"Current working directory: {Path.cwd()}")

DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_sciebo_public_file(share_url: str, password: str, output_path: Path) -> None:
    """Download a password-protected public sciebo/ownCloud share via WebDAV."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and zipfile.is_zipfile(output_path):
        print(f"{output_path} already exists and is a valid zip file. Skipping download.")
        return

    parsed = urlparse(share_url)
    base_url = f"{parsed.scheme}://{parsed.netloc}"
    token = parsed.path.rstrip("/").split("/")[-1]
    webdav_url = f"{base_url}/public.php/webdav/"

    tmp_path = output_path.with_suffix(output_path.suffix + ".part")

    print("Downloading dataset from sciebo...")

    with requests.get(
        webdav_url,
        auth=(token, password),
        stream=True,
        timeout=60,
    ) as response:
        response.raise_for_status()

        total_bytes = int(response.headers.get("content-length", 0))

        with tmp_path.open("wb") as f, tqdm(
            total=total_bytes if total_bytes > 0 else None,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc="Download",
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    progress.update(len(chunk))

    tmp_path.replace(output_path)

    if not zipfile.is_zipfile(output_path):
        raise RuntimeError(
            f"Downloaded file at {output_path} is not a valid zip file. "
            "Please check that the sciebo link points directly to the dataset zip."
        )

    print(f"Downloaded dataset to {output_path}")


def verify_dataset_dirs() -> None:
    missing_dirs = [path for path in EXPECTED_DATASET_DIRS if not path.is_dir()]

    if missing_dirs:
        missing = "\n".join(f"  - {path}" for path in missing_dirs)
        raise FileNotFoundError(
            "Dataset extraction did not create the expected folders:\n"
            f"{missing}\n\n"
            "Please check the zip file structure. The expected folders must exist directly under data/."
        )

    print("Dataset verified. Found expected folders:")
    for path in EXPECTED_DATASET_DIRS:
        print(f"  - {path.relative_to(Path.cwd())}")


if all(path.is_dir() for path in EXPECTED_DATASET_DIRS):
    print("Expected dataset folders already exist. Skipping download and extraction.")
else:
    download_sciebo_public_file(
        share_url=SCIEBO_URL,
        password=SCIEBO_PASSWORD,
        output_path=ZIP_PATH,
    )

    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print(f"Extracted dataset into {DATA_DIR.relative_to(Path.cwd())}/")


verify_dataset_dirs()

try:
    ZIP_PATH.unlink(missing_ok=True)
    print(f"Deleted zip file: {ZIP_PATH.relative_to(Path.cwd())}")
except OSError as error:
    print(f"Warning: could not delete zip file {ZIP_PATH}: {error}")

print("Done!")

## Connection to previous exercises

This project builds directly on the earlier notebooks and solutions:

- **Convolution kernels:** You saw how local filters respond to edges, blur, and texture. A CNN learns such filters from data.
- **CNN from scratch on a real dataset:** You implemented convolutional blocks, trained a CNN, plotted curves, and evaluated predictions.
- **MVTec / industrial transfer learning:** You saw that industrial inspection requires more than accuracy, especially when classes are imbalanced.
- **CIFAR-10 transfer learning:** You saw a complete training/evaluation workflow, although pretrained models are not allowed in this first part.
- **YOLO detection/segmentation:** This showed broader industrial CV tasks. In this project, however, we focus on image-level classification.

You may use the previous solutions as references, but you must adapt the ideas to this dataset and justify your decisions.

## Expected dataset

The clean student dataset should have the following structure:

```text
data/
  hazelnut_binary_raw/
    good/
    defective/
    metadata.csv
    class_to_label.csv
```

You should create your own train/validation/test split from this raw dataset.

**Important:**  
Use a fixed random seed and make sure the split is stratified, meaning each split should contain both `good` and `defective` samples.

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

# TODO: add further imports when needed
# Small hints: random, json/csv, numpy, pandas, matplotlib,
# PIL, torchvision, sklearn

## 1. Configuration

Set paths, random seed, image size, batch size, learning rate, and other hyperparameters.

These choices must be documented in your report.

In [ ]:
DATA_ROOT = Path("data/hazelnut_binary_raw")

SEED = 42

# TODO: choose suitable values and justify them in your report
IMAGE_SIZE = None
BATCH_SIZE = None
LEARNING_RATE = None
NUM_EPOCHS = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# TODO: make your experiment reproducible
# Hint: set seeds for random, numpy, and torch

## 2. Dataset inspection (Optional, but helpful)

Load `metadata.csv` and inspect the dataset.

Questions to answer:

- How many good and defective images are available?
- Is the dataset balanced?
- What are the consequences for training and evaluation?
- What do typical good and defective images look like?

Recall from the MVTec transfer-learning exercise: in industrial inspection, class imbalance is common and accuracy alone can be misleading.

In [ ]:
# TODO: load metadata.csv
metadata = None

# TODO: visualize example images from both classes

## 3. Create your own train/validation/test split

Create a reproducible train/validation/test split and save it as `split.csv`.

Requirements:

- Use a fixed random seed.
- Use a stratified split.
- Report the class distribution for each split.
- Briefly justify your chosen split in the report.

The exported `split.csv` will be reused in Part 2 so that the scratch CNN and fine-tuned model are evaluated on the same images.

In [ ]:
# TODO: create a stratified train/validation/test split

# Expected output:
# split_df columns:
# - relative_path
# - label
# - class_name
# - split   ("train", "val", "test")

split_df = None


def export_split_csv(split_df, output_path="split.csv"):
    """
    Validate and save the train/validation/test split.

    The saved file is used again in Notebook 02 for transfer learning and
    corruption evaluation. Keeping this file fixed makes the experiments
    reproducible and makes model comparisons fair.
    """
    output_path = Path(output_path)
    required_columns = {"relative_path", "label", "class_name", "split"}
    valid_splits = {"train", "val", "test"}

    missing_columns = required_columns - set(split_df.columns)
    if missing_columns:
        raise ValueError(f"split_df is missing required columns: {sorted(missing_columns)}")

    unknown_splits = set(split_df["split"].unique()) - valid_splits
    if unknown_splits:
        raise ValueError(f"Unknown split names: {sorted(unknown_splits)}")

    for split_name in sorted(valid_splits):
        sub = split_df[split_df["split"] == split_name]
        if sub.empty:
            raise ValueError(f"Split '{split_name}' is empty.")

        labels = set(sub["label"].unique())
        if labels != {0, 1}:
            raise ValueError(
                f"Split '{split_name}' should contain both labels 0 and 1, "
                f"but contains labels: {sorted(labels)}"
            )

    split_df.to_csv(output_path, index=False)
    print(f"Saved split to: {output_path}")

    print("Class distribution per split:")
    print(split_df.groupby(["split", "class_name"]).size())


# TODO: after creating split_df, uncomment this line:
# export_split_csv(split_df, "split.csv")

## 4. Define dataset and dataloaders

Implement a PyTorch `Dataset`.

Think carefully about:

- loading images,
- converting labels,
- resizing images,
- normalization,
- whether training and evaluation transforms should differ.

In [ ]:
class HazelnutDataset(Dataset):
    def __init__(self, root_dir, split_df, split_name, transform=None):
        # TODO: implement
        pass

    def __len__(self):
        # TODO: implement
        pass

    def __getitem__(self, idx):
        # TODO: implement
        pass


# TODO: define train/validation/test transforms
train_transform = None
eval_transform = None

# TODO: create datasets
train_dataset = None
val_dataset = None
test_dataset = None

# TODO: create dataloaders
train_loader = None
val_loader = None
test_loader = None

## 5. Design your CNN from scratch

In the convolution-kernel exercise, you applied manually designed filters such as edge and blur kernels.  
In a CNN, these filters are learned from data.

Your CNN should contain convolutional layers that learn local visual patterns and deeper layers that combine them into more abstract features.

You may use standard PyTorch layers such as:

- `nn.Conv2d`
- `nn.ReLU`
- `nn.MaxPool2d`
- `nn.BatchNorm2d`
- `nn.Dropout`
- `nn.Linear`

You may **not** use pretrained models in this first part of the practical project.

In [ ]:
# TODO: optionally define a reusable convolutional block.
# Hint from the previous CNN exercise:
# Conv2d -> ReLU -> MaxPool2d
# You may extend this with BatchNorm or Dropout if justified.

class ScratchCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: define your architecture
        pass

    def forward(self, x):
        # TODO: implement forward pass
        pass


model = ScratchCNN().to(DEVICE)
print(model)

# TODO: count trainable parameters

## 6. Loss function, optimizer, and training loop

Important questions:

- Which loss function is appropriate?
- How do you handle class imbalance?
- Which optimizer do you use?
- How do you monitor overfitting?
- Which validation metric do you use for model selection?

Recall: a defective image predicted as good is a **missed defect**. In many industrial settings, this is more critical than a false alarm.

In [ ]:
# TODO: define criterion and optimizer
criterion = None
optimizer = None

# TODO: consider class weighting or another strategy for class imbalance

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    # TODO: implement one training epoch
    pass


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    # TODO: implement validation/test evaluation
    # Suggested return values: loss, accuracy, y_true, y_pred, probabilities
    pass


history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
}

# TODO: training loop

## 7. Plot training curves

Plot at least:

- training loss,
- validation loss,
- training accuracy,
- validation accuracy.


In [ ]:
# TODO: plot training/validation curves

## 8. Export the final model for reproducible teacher-side evaluation

For the official hidden-test evaluation, the teaching team must be able to load your submitted model **without knowing your Python class definition**.

Therefore, submit a TorchScript model saved as:

```text
model.pt
```

The helper function below exports your trained model as TorchScript and immediately reloads it to check that the saved file can actually run again.


In [ ]:
import json


def export_torchscript_model(
    model: torch.nn.Module,
    path: str | Path,
    image_size: int,
    device: torch.device,
    metadata_path: str | Path | None = "model_metadata.json",
    extra_info: dict | None = None,
) -> torch.jit.ScriptModule:
    """
    Export the trained model as TorchScript.

    Why TorchScript?
    ----------------
    For the hidden-test evaluation, the teaching team should be able to load
    the submitted model without knowing the exact Python class definition that
    each group used for ScratchCNN.

    A normal state_dict only stores weights. It does not store the architecture.
    TorchScript stores an executable representation of the model, which makes
    the submitted model.pt much easier to evaluate reproducibly.

    Assumptions
    -----------
    - Input shape:  (N, 3, image_size, image_size)
    - Output shape: (N, 2)
    - Class labels: 0 = good, 1 = defective
    """
    path = Path(path)
    model.eval()
    model.to(device)

    if image_size is None:
        raise ValueError("IMAGE_SIZE must be set before exporting the model.")

    example_input = torch.zeros(1, 3, int(image_size), int(image_size), device=device)

    # Tracing is suitable here because the CNN should use a standard feed-forward
    # forward pass without data-dependent control flow.
    traced_model = torch.jit.trace(model, example_input)

    # Basic sanity check: the exported model should produce two logits.
    with torch.no_grad():
        example_output = traced_model(example_input)

    if example_output.ndim != 2 or example_output.shape[1] != 2:
        raise ValueError(
            "The exported model must return logits with shape (batch_size, 2). "
            f"Got output shape: {tuple(example_output.shape)}"
        )

    traced_model.save(str(path))
    print(f"Saved TorchScript model to: {path}")

    metadata = {
        "format": "torchscript",
        "image_size": int(image_size),
        "input_shape": ["batch_size", 3, int(image_size), int(image_size)],
        "output_shape": ["batch_size", 2],
        "class_to_label": {"good": 0, "defective": 1},
        "label_to_class": {"0": "good", "1": "defective"},
    }

    if extra_info is not None:
        metadata["extra_info"] = extra_info

    if metadata_path is not None:
        metadata_path = Path(metadata_path)
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        print(f"Saved optional metadata to: {metadata_path}")

    return traced_model


@torch.no_grad()
def verify_exported_torchscript_model(
    model_path: str | Path,
    image_size: int,
    device: torch.device,
) -> torch.jit.ScriptModule:
    """
    Reload the submitted model.pt and verify that it can run independently.
    """
    model_path = Path(model_path)
    loaded_model = torch.jit.load(str(model_path), map_location=device)
    loaded_model.eval()

    test_input = torch.zeros(2, 3, int(image_size), int(image_size), device=device)
    test_output = loaded_model(test_input)

    if test_output.shape != (2, 2):
        raise ValueError(
            "Reloaded model should return logits with shape (2, 2), "
            f"but returned {tuple(test_output.shape)}."
        )

    print("Reload check successful.")
    print(f"Input shape:  {tuple(test_input.shape)}")
    print(f"Output shape: {tuple(test_output.shape)}")
    return loaded_model


submission_model_path = Path("model.pt")

exported_model = export_torchscript_model(
    model=model,
    path=submission_model_path,
    image_size=IMAGE_SIZE,
    device=DEVICE,
    metadata_path="model_metadata.json",  # useful for your report, but not part of the required upload
    extra_info={
        "model_type": "ScratchCNN",
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        # Optional TODO: add any other relevant information, e.g. architecture notes
    },
)

reloaded_model = verify_exported_torchscript_model(
    model_path=submission_model_path,
    image_size=IMAGE_SIZE,
    device=DEVICE,
)

# TODO: evaluate reloaded_model on the validation or test set.
# The results should match the original model up to small numerical differences.


## 9. Local minimum-performance check on your test split

Before submitting your model, evaluate it on your own test split. This gives you an estimate of whether the model is likely to pass the first part of the practical project.

The official score will be computed by the teaching team on a hidden test set. Your local test score is therefore only a guide, not the final score.

The final evaluation is based on the following key metrics:

- recall for the `defective` class,
- F1-score for the `defective` class.

Accuracy alone is not sufficient because the dataset is imbalanced.

After checking your local result, upload your final `model.pt` to the Sciebo submission folder mentioned in the initial project presentation. The teaching team will evaluate the uploaded model on the hidden test set and provide the official model scores.

In [ ]:
MIN_RECALL_DEFECTIVE = 0.80
MIN_F1_DEFECTIVE = 0.70


@torch.no_grad()
def evaluate_minimum_criteria(
    model: torch.nn.Module,
    loader: DataLoader,
    device: torch.device,
    min_recall_defective: float = MIN_RECALL_DEFECTIVE,
    min_f1_defective: float = MIN_F1_DEFECTIVE,
    plot_confusion_matrix: bool = True,
) -> dict[str, float]:
    """
    Evaluate the model on a local test set and check the minimum criteria.

    This function assumes a two-class classifier with labels:
        0 = good
        1 = defective

    The official score is computed separately on the hidden teacher test set.
    """
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        confusion_matrix,
        ConfusionMatrixDisplay,
        classification_report,
    )

    model.eval()
    y_true = []
    y_pred = []

    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy()

        y_pred.extend(predictions.tolist())
        y_true.extend(labels.cpu().numpy().tolist())

    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    print(classification_report(
        y_true,
        y_pred,
        target_names=["good", "defective"],
        zero_division=0,
    ))

    print("Minimum local criteria")
    print("----------------------")
    print(f"Defective recall: {recall:.3f} / required: {min_recall_defective:.3f}")
    print(f"Defective F1:     {f1:.3f} / required: {min_f1_defective:.3f}")

    passed_recall = recall >= min_recall_defective
    passed_f1 = f1 >= min_f1_defective
    passed = passed_recall and passed_f1

    if passed:
        print("Local check: PASSED")
    else:
        print("Local check: NOT YET PASSED")
        if not passed_recall:
            print("- Defective recall is below the required value.")
        if not passed_f1:
            print("- Defective F1-score is below the required value.")

    if plot_confusion_matrix:
        cm = confusion_matrix(y_true, y_pred)
        disp = ConfusionMatrixDisplay(cm, display_labels=["good", "defective"])
        disp.plot()
        plt.title("Confusion matrix on local test split")
        plt.show()

    return {
        "accuracy": accuracy,
        "precision_defective": precision,
        "recall_defective": recall,
        "f1_defective": f1,
        "passed_local_minimum": passed,
    }


# TODO: run this after test_loader and your trained model are available.
# local_test_results = evaluate_minimum_criteria(model, test_loader, DEVICE)
# local_test_results

## 10. Failure analysis

Visualize and discuss examples of:

- false positives: good image predicted as defective,
- false negatives: defective image predicted as good.

In your report, explain what these mistakes would mean in an industrial inspection system.

In [ ]:
# TODO: visualize false positives and false negatives

## 11. Official hidden-test evaluation and submission

The local check above is only an estimate. The official model score for the CNN will be computed by the teaching team on a hidden test set.

Submit a group folder containing exactly:

```text
model.pt
members.txt
```

`model.pt` must be the TorchScript model exported in Section 8. Do **not** submit only a `state_dict`, because the teaching team cannot reliably reconstruct every group's custom `ScratchCNN` architecture from weights alone.

`members.txt` should contain the group members separated by newlines, for example:

```text
Max Mustermann
Erika Musterfrau
Alex Beispiel
```

Your report should include:

- your local test-set recall and F1-score for the `defective` class,
- the confusion matrix on your own test split,
- a short discussion of whether your local result satisfies the minimum criterion,
- a reminder that the official score is based on the hidden test set.
